## 📦 1. Configuration et Imports

In [1]:
# ============================================================
# IMPORTS ET CONFIGURATION
# ============================================================

import pandas as pd
import numpy as np
import os
import re
from datetime import datetime
from typing import Dict, List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# Tentative d'import de pdfplumber pour l'extraction PDF
try:
    import pdfplumber
    PDF_SUPPORT = True
    print("✅ pdfplumber disponible - Extraction PDF activée")
except ImportError:
    PDF_SUPPORT = False
    print("⚠️ pdfplumber non disponible - Installation requise: pip install pdfplumber")

# Configuration des chemins
BASE_PATH = '/home/henintsoa/CFIM'
METEO_PATH = os.path.join(BASE_PATH, 'meteo')
CSV_PATH = os.path.join(BASE_PATH, 'csv')
OUTPUT_PATH = os.path.join(BASE_PATH, 'final/data')

print(f"\n📂 Chemins configurés:")
print(f"   • Base: {BASE_PATH}")
print(f"   • Météo: {METEO_PATH}")
print(f"   • CSV: {CSV_PATH}")
print(f"   • Output: {OUTPUT_PATH}")

✅ pdfplumber disponible - Extraction PDF activée

📂 Chemins configurés:
   • Base: /home/henintsoa/CFIM
   • Météo: /home/henintsoa/CFIM/meteo
   • CSV: /home/henintsoa/CFIM/csv
   • Output: /home/henintsoa/CFIM/final/data


## 📁 2. Inventaire des Bulletins Disponibles

Avant d'extraire les données, faisons un **inventaire complet** de tous les bulletins météo disponibles par année.

In [2]:
# ============================================================
# INVENTAIRE DES DOSSIERS METEO PAR ANNÉE
# ============================================================

def inventorier_bulletins_meteo(meteo_path: str) -> pd.DataFrame:
    """
    Parcourt tous les dossiers météo et compte les bulletins disponibles.
    
    Returns:
        DataFrame avec le nombre de bulletins par année et type
    """
    inventaire = []
    
    # Parcourir les années
    for year_folder in sorted(os.listdir(meteo_path)):
        year_path = os.path.join(meteo_path, year_folder)
        
        if not os.path.isdir(year_path) or not year_folder.startswith('METEO'):
            continue
        
        # Extraire l'année
        year_match = re.search(r'(\d{4})', year_folder)
        if not year_match:
            continue
        year = year_match.group(1)
        
        # Compter les sous-dossiers (jours)
        nb_jours = 0
        nb_pdf_marine_cotiere = 0
        nb_pdf_marine_hm = 0
        nb_pdf_total = 0
        
        for day_folder in os.listdir(year_path):
            day_path = os.path.join(year_path, day_folder)
            
            if not os.path.isdir(day_path):
                continue
            
            nb_jours += 1
            
            # Compter les PDF
            for file in os.listdir(day_path):
                if file.endswith('.pdf'):
                    nb_pdf_total += 1
                    if 'cotiere' in file.lower():
                        nb_pdf_marine_cotiere += 1
                    elif 'marine' in file.lower() and 'hm' in file.lower():
                        nb_pdf_marine_hm += 1
        
        inventaire.append({
            'Année': year,
            'Jours disponibles': nb_jours,
            'PDF Marine Côtière': nb_pdf_marine_cotiere,
            'PDF Marine HM': nb_pdf_marine_hm,
            'Total PDF': nb_pdf_total
        })
    
    return pd.DataFrame(inventaire)

# Exécuter l'inventaire
df_inventaire = inventorier_bulletins_meteo(METEO_PATH)

print("📊 INVENTAIRE DES BULLETINS MÉTÉO DISPONIBLES")
print("=" * 70)
display(df_inventaire)

# Statistiques globales
total_jours = df_inventaire['Jours disponibles'].sum()
total_pdf_cotiere = df_inventaire['PDF Marine Côtière'].sum()
print(f"\n📈 TOTAL:")
print(f"   • {total_jours} jours de données")
print(f"   • {total_pdf_cotiere} bulletins Marine Côtière à extraire")

📊 INVENTAIRE DES BULLETINS MÉTÉO DISPONIBLES


,Année,Jours disponibles,PDF Marine Côtière,PDF Marine HM,Total PDF
0,2019,103,177,60,334
1,2020,88,131,19,295
2,2021,161,229,147,518
3,2022,181,253,184,801
4,2023,133,183,85,369
5,2024,0,0,0,0
6,2025,23,24,27,83



📈 TOTAL:
   • 689 jours de données
   • 997 bulletins Marine Côtière à extraire


## 🔧 3. Classe d'Extraction des Bulletins Météo

Cette classe contient toute la logique pour :
1. **Lire les fichiers PDF** des bulletins marins
2. **Parser le texte** pour extraire les informations structurées
3. **Identifier les zones côtières** et leurs conditions météo

In [3]:
# ============================================================
# CLASSE D'EXTRACTION DES BULLETINS MÉTÉO
# ============================================================

class BulletinMeteoExtractor:
    """
    Extracteur de données météorologiques marines depuis les bulletins PDF.
    
    Cette classe parse les bulletins Marine Côtière de Madagascar et extrait:
    - Les zones côtières concernées
    - Les conditions de vent (direction, vitesse)
    - L'état de la mer
    - Les conditions météo générales
    """
    
    def __init__(self):
        # ========================================
        # ZONES CÔTIÈRES DE MADAGASCAR
        # ========================================
        # Ces zones sont utilisées dans les bulletins marins
        self.zones_cotieres = [
            "CAP D'AMBRE A TOAMASINA",
            "CAP D'AMBRE A MAHANORO",
            "CAP D'AMBRE A ANTALAHA",
            "TOAMASINA AU CAP SAINTE MARIE",
            "TOAMASINA A TAOLAGNARO",
            "MAHANORO AU CAP SAINTE MARIE",
            "CAP D'AMBRE A BESALAMPY",
            "BESALAMPY A MOROMBE",
            "MOROMBE AU CAP SAINTE MARIE",
            "MOROMBE A TAOLAGNARO",
            "CAP D'AMBRE A CAP EST",
            "CAP EST A TAOLAGNARO"
        ]
        
        # ========================================
        # PATTERNS REGEX POUR L'EXTRACTION
        # ========================================
        
        # Pattern pour les sections de zone
        self.pattern_section = r"([A-Z\s'ÀÉÈÊ\-]+(?:A|AU)\s+[A-Z\s'ÀÉÈÊ\-]+)\s*\n"
        
        # Pattern pour le vent
        self.pattern_vent = r"VENT\s*[:\s]*(.+?)(?=ETAT|MER|TEMPS|HOULE|$)"
        
        # Pattern pour l'état de la mer
        self.pattern_mer = r"(?:ETAT DE LA MER|MER)\s*[:\s]*(.+?)(?=TEMPS|HOULE|VENT|$)"
        
        # Pattern pour le temps
        self.pattern_temps = r"TEMPS\s*[:\s]*(.+?)(?=VENT|ETAT|MER|HOULE|$)"
        
        # Pattern pour la houle
        self.pattern_houle = r"HOULE\s*[:\s]*(.+?)(?=VENT|ETAT|MER|TEMPS|$)"
    
    def extraire_date_depuis_nom(self, nom_fichier: str) -> Optional[str]:
        """
        Extrait la date depuis le nom du fichier ou du dossier.
        
        Formats supportés:
        - DDMMYYYY (ex: 01032022)
        - YYYY-MM-DD (ex: 2019-09-03)
        
        Returns:
            Date au format DD/MM/YYYY ou None
        """
        # Format DDMMYYYY
        match = re.search(r'(\d{2})(\d{2})(\d{4})', nom_fichier)
        if match:
            jour, mois, annee = match.groups()
            return f"{jour}/{mois}/{annee}"
        
        # Format YYYY-MM-DD
        match = re.search(r'(\d{4})-(\d{2})-(\d{2})', nom_fichier)
        if match:
            annee, mois, jour = match.groups()
            return f"{jour}/{mois}/{annee}"
        
        return None
    
    def lire_pdf(self, pdf_path: str) -> Optional[str]:
        """
        Lit le contenu textuel d'un fichier PDF.
        
        Args:
            pdf_path: Chemin vers le fichier PDF
            
        Returns:
            Texte extrait du PDF ou None en cas d'erreur
        """
        if not PDF_SUPPORT:
            return None
        
        try:
            with pdfplumber.open(pdf_path) as pdf:
                texte_complet = ""
                for page in pdf.pages:
                    texte = page.extract_text()
                    if texte:
                        texte_complet += texte + "\n"
                return texte_complet
        except Exception as e:
            print(f"   ❌ Erreur lecture PDF {pdf_path}: {e}")
            return None
    
    def lire_texte_extrait(self, folder_path: str) -> Optional[str]:
        """
        Lit le texte depuis le dossier extracted_text/ si disponible.
        
        Args:
            folder_path: Chemin vers le dossier du jour
            
        Returns:
            Texte extrait ou None
        """
        extracted_path = os.path.join(folder_path, 'extracted_text')
        
        if not os.path.exists(extracted_path):
            return None
        
        texte_complet = ""
        for file in os.listdir(extracted_path):
            if file.endswith('.txt'):
                with open(os.path.join(extracted_path, file), 'r', encoding='utf-8', errors='ignore') as f:
                    texte_complet += f.read() + "\n"
        
        return texte_complet if texte_complet else None
    
    def parser_bulletin(self, texte: str, date_str: str) -> List[Dict]:
        """
        Parse le texte d'un bulletin météo et extrait les données par zone.
        
        Args:
            texte: Texte du bulletin
            date_str: Date au format DD/MM/YYYY
            
        Returns:
            Liste de dictionnaires avec les données par zone
        """
        resultats = []
        
        # Nettoyer le texte
        texte = texte.upper()
        texte = re.sub(r'\s+', ' ', texte)
        
        # Pattern amélioré pour matcher les sections
        # Cherche les blocs: ZONE + VENT + ETAT DE LA MER + TEMPS
        pattern_bloc = r"([A-Z'\s]+(?:A|AU)\s+[A-Z'\s]+)\s*" + \
                      r"(?:VENT\s*[:\s]*([^E]+?))?" + \
                      r"(?:ETAT DE LA MER\s*[:\s]*([^T]+?))?" + \
                      r"(?:TEMPS\s*[:\s]*([^V]+?))?"
        
        # Approche alternative: split par zone
        for zone in self.zones_cotieres:
            zone_upper = zone.upper()
            
            # Chercher la zone dans le texte
            if zone_upper in texte:
                # Trouver la position de la zone
                pos = texte.find(zone_upper)
                
                # Extraire le bloc suivant (jusqu'à la prochaine zone ou fin)
                end_pos = len(texte)
                for autre_zone in self.zones_cotieres:
                    if autre_zone.upper() != zone_upper:
                        autre_pos = texte.find(autre_zone.upper(), pos + len(zone_upper))
                        if autre_pos > pos and autre_pos < end_pos:
                            end_pos = autre_pos
                
                bloc = texte[pos:end_pos]
                
                # Extraire les informations du bloc
                vent = self._extraire_champ(bloc, r"VENT\s*[:\s]*(.+?)(?=ETAT|MER|TEMPS|HOULE|[A-Z']+\s+A\s|$)")
                etat_mer = self._extraire_champ(bloc, r"(?:ETAT DE LA MER|MER)\s*[:\s]*(.+?)(?=TEMPS|HOULE|VENT|[A-Z']+\s+A\s|$)")
                temps = self._extraire_champ(bloc, r"TEMPS\s*[:\s]*(.+?)(?=VENT|ETAT|MER|HOULE|[A-Z']+\s+A\s|$)")
                houle = self._extraire_champ(bloc, r"HOULE\s*[:\s]*(.+?)(?=VENT|ETAT|MER|TEMPS|[A-Z']+\s+A\s|$)")
                
                if vent or etat_mer or temps:
                    resultats.append({
                        'date': date_str,
                        'zone': zone,
                        'vent': vent if vent else 'Non spécifié',
                        'etat_mer': etat_mer if etat_mer else 'Non spécifié',
                        'temps': temps if temps else 'Non spécifié',
                        'houle': houle if houle else 'Non spécifié'
                    })
        
        return resultats
    
    def _extraire_champ(self, texte: str, pattern: str) -> Optional[str]:
        """
        Extrait un champ spécifique du texte avec un pattern regex.
        """
        match = re.search(pattern, texte, re.IGNORECASE | re.DOTALL)
        if match:
            valeur = match.group(1).strip()
            # Nettoyer la valeur
            valeur = re.sub(r'\s+', ' ', valeur)
            valeur = valeur.strip('., ')
            return valeur[:200]  # Limiter la longueur
        return None

print("✅ Classe BulletinMeteoExtractor chargée")

✅ Classe BulletinMeteoExtractor chargée


## 🔄 4. Extraction Massive des Bulletins

Maintenant, nous allons parcourir **tous les dossiers météo** et extraire les données de chaque bulletin.

In [4]:
# ============================================================
# FONCTION D'EXTRACTION GLOBALE
# ============================================================

def extraire_tous_bulletins(meteo_path: str, annees: List[str] = None) -> pd.DataFrame:
    """
    Extrait les données de tous les bulletins météo disponibles.
    
    Args:
        meteo_path: Chemin vers le dossier meteo/
        annees: Liste des années à traiter (None = toutes)
        
    Returns:
        DataFrame avec toutes les données extraites
    """
    extracteur = BulletinMeteoExtractor()
    tous_resultats = []
    
    stats = {
        'dossiers_traites': 0,
        'pdf_lus': 0,
        'textes_extraits': 0,
        'zones_extraites': 0,
        'erreurs': 0
    }
    
    # Parcourir les années
    for year_folder in sorted(os.listdir(meteo_path)):
        year_path = os.path.join(meteo_path, year_folder)
        
        if not os.path.isdir(year_path) or not year_folder.startswith('METEO'):
            continue
        
        # Filtrer par année si spécifié
        year_match = re.search(r'(\d{4})', year_folder)
        if year_match:
            year = year_match.group(1)
            if annees and year not in annees:
                continue
        
        print(f"\n📅 Traitement de {year_folder}...")
        
        # Parcourir les jours
        for day_folder in sorted(os.listdir(year_path)):
            day_path = os.path.join(year_path, day_folder)
            
            if not os.path.isdir(day_path):
                continue
            
            stats['dossiers_traites'] += 1
            
            # Extraire la date
            date_str = extracteur.extraire_date_depuis_nom(day_folder)
            if not date_str:
                continue
            
            texte = None
            
            # Méthode 1: Lire le texte déjà extrait
            texte = extracteur.lire_texte_extrait(day_path)
            if texte:
                stats['textes_extraits'] += 1
            
            # Méthode 2: Lire directement le PDF
            if not texte and PDF_SUPPORT:
                for file in os.listdir(day_path):
                    if 'cotiere' in file.lower() and file.endswith('.pdf'):
                        pdf_path = os.path.join(day_path, file)
                        texte = extracteur.lire_pdf(pdf_path)
                        if texte:
                            stats['pdf_lus'] += 1
                            break
            
            # Parser le bulletin
            if texte:
                try:
                    resultats = extracteur.parser_bulletin(texte, date_str)
                    tous_resultats.extend(resultats)
                    stats['zones_extraites'] += len(resultats)
                except Exception as e:
                    stats['erreurs'] += 1
        
        print(f"   ✓ {stats['dossiers_traites']} dossiers traités jusqu'ici")
    
    # Afficher les statistiques
    print("\n" + "=" * 70)
    print("📊 STATISTIQUES D'EXTRACTION")
    print("=" * 70)
    for key, value in stats.items():
        print(f"   • {key.replace('_', ' ').title()}: {value}")
    
    return pd.DataFrame(tous_resultats)

print("✅ Fonction d'extraction globale prête")

✅ Fonction d'extraction globale prête


In [5]:
# ============================================================
# EXÉCUTION DE L'EXTRACTION
# ============================================================

print("🚀 DÉMARRAGE DE L'EXTRACTION DES BULLETINS MÉTÉO")
print("=" * 70)
print("Cette opération peut prendre plusieurs minutes...\n")

# Extraire les données de 2019 à 2022
# (2017-2018 ne semblent pas avoir de données structurées)
df_meteo_extrait = extraire_tous_bulletins(METEO_PATH, annees=['2019', '2020', '2021', '2022'])

print(f"\n✅ Extraction terminée!")
print(f"📊 {len(df_meteo_extrait)} enregistrements extraits")

🚀 DÉMARRAGE DE L'EXTRACTION DES BULLETINS MÉTÉO
Cette opération peut prendre plusieurs minutes...


📅 Traitement de METEO 2019...
   ✓ 103 dossiers traités jusqu'ici

📅 Traitement de METEO 2020...
   ✓ 191 dossiers traités jusqu'ici

📅 Traitement de METEO 2021...
   ✓ 352 dossiers traités jusqu'ici

📅 Traitement de METEO 2022...
   ✓ 533 dossiers traités jusqu'ici

📊 STATISTIQUES D'EXTRACTION
   • Dossiers Traites: 533
   • Pdf Lus: 495
   • Textes Extraits: 1
   • Zones Extraites: 886
   • Erreurs: 0

✅ Extraction terminée!
📊 886 enregistrements extraits


## 📊 5. Aperçu des Données Extraites

In [6]:
# ============================================================
# APERÇU DES DONNÉES EXTRAITES
# ============================================================

if len(df_meteo_extrait) > 0:
    print("📋 APERÇU DES DONNÉES EXTRAITES")
    print("=" * 70)
    
    print(f"\n📈 Dimensions: {df_meteo_extrait.shape[0]} lignes × {df_meteo_extrait.shape[1]} colonnes")
    print(f"\n📋 Colonnes: {list(df_meteo_extrait.columns)}")
    
    print("\n📝 Échantillon des données:")
    display(df_meteo_extrait.head(15))
    
    # Statistiques par zone
    print("\n🗺️ RÉPARTITION PAR ZONE CÔTIÈRE")
    print("-" * 50)
    print(df_meteo_extrait['zone'].value_counts())
    
    # Statistiques temporelles
    df_meteo_extrait['date_parsed'] = pd.to_datetime(df_meteo_extrait['date'], format='%d/%m/%Y', errors='coerce')
    df_meteo_extrait['annee'] = df_meteo_extrait['date_parsed'].dt.year
    
    print("\n📅 RÉPARTITION PAR ANNÉE")
    print("-" * 50)
    print(df_meteo_extrait['annee'].value_counts().sort_index())
else:
    print("⚠️ Aucune donnée extraite.")
    print("Vérifiez que pdfplumber est installé: pip install pdfplumber")

📋 APERÇU DES DONNÉES EXTRAITES

📈 Dimensions: 886 lignes × 6 colonnes

📋 Colonnes: ['date', 'zone', 'vent', 'etat_mer', 'temps', 'houle']

📝 Échantillon des données:


,date,zone,vent,etat_mer,temps,houle
0,07/09/2019,CAP D'AMBRE A MAHANORO,DE SUD-EST 15/20 KT ATTEIGNANT 25/30 KT AU NOR...,AGITÉE À FORTE. HAUTEUR DE VAGUE 2.8/3.2M,Non spécifié,DE SUD-EST
1,07/09/2019,MAHANORO AU CAP SAINTE MARIE,DE SUD-EST 20/25 KT ATTEIGNANT LOCALEMENT 30KT,"AGITÉE À FORTE, TRES FORTE PAR",Non spécifié,M
2,07/09/2019,CAP D'AMBRE A BESALAMPY,DE SUD-EST 15/20 KT ATTEIGNANT 30 KT LE MATIN ...,PEU AGITÉ,Non spécifié,Non spécifié
3,07/09/2019,BESALAMPY A MOROMBE,DE SECTEUR SUD 20/25 KT. LOCALEMENT 30 KT AU C...,AGITÉ,SEC,M
4,07/09/2019,MOROMBE AU CAP SAINTE MARIE,DE SUD-EST 20/25 KT TOURNANT SECTEUR EST TEMPO...,F,PARTIELLEMENT NUAGEUX,M
5,06/09/2019,CAP D'AMBRE A MAHANORO,DE SUD-EST 10/15 KT ATTEIGNANT 20/25 KT AU NOR...,AGITÉE À FORTE. HAUTEUR DE VAGUE 2/2.5M,Non spécifié,DE SUD-EST
6,06/09/2019,MAHANORO AU CAP SAINTE MARIE,DE NORD 05/10 KT DEVENANT PROGRESSIVEMENT SECT...,AGITÉE À FORTE. HAUTEUR DE VAGUE 2/2.5M ATTEIG...,Non spécifié,MODEREE DE SUD-OUEST
7,06/09/2019,CAP D'AMBRE A BESALAMPY,DE SUD 15/20 KT LOCALEMENT 20 KT AU SUD DE MAJ...,PEU AGITÉ,SEC,Non spécifié
8,06/09/2019,BESALAMPY A MOROMBE,DE SECTEUR SUD 15/20 KT,AGITÉ,SEC,M
9,09/09/2019,CAP D'AMBRE A MAHANORO,DE SUD-EST 05/10 KT LOCALEMENT 15 KT AU SUD D'...,FORTE. HAUTEUR DE VAGUE 2/3.5M,Non spécifié,MODÉRÉE DE SUD-EST



🗺️ RÉPARTITION PAR ZONE CÔTIÈRE
--------------------------------------------------
zone
CAP D'AMBRE A CAP EST            351
CAP D'AMBRE A BESALAMPY          143
BESALAMPY A MOROMBE              123
MOROMBE A TAOLAGNARO              54
CAP D'AMBRE A MAHANORO            53
MOROMBE AU CAP SAINTE MARIE       48
CAP D'AMBRE A ANTALAHA            43
MAHANORO AU CAP SAINTE MARIE      32
CAP D'AMBRE A TOAMASINA           19
TOAMASINA A TAOLAGNARO            12
TOAMASINA AU CAP SAINTE MARIE      7
CAP EST A TAOLAGNARO               1
Name: count, dtype: int64

📅 RÉPARTITION PAR ANNÉE
--------------------------------------------------
annee
2019    373
2020    200
2021    142
2022    171
Name: count, dtype: int64


## 📊 6. Consolidation avec les Données Existantes

Nous allons maintenant **fusionner** les nouvelles données extraites avec le fichier CSV existant `marine_cotiere_2019_2020.csv`.

In [7]:
# ============================================================
# CHARGER LES DONNÉES EXISTANTES
# ============================================================

# Charger le fichier CSV existant
csv_existant = os.path.join(CSV_PATH, 'marine_cotiere_2019_2020.csv')
df_existant = pd.read_csv(csv_existant)

print("📂 DONNÉES EXISTANTES")
print("=" * 70)
print(f"Fichier: {csv_existant}")
print(f"Nombre d'enregistrements: {len(df_existant)}")
print(f"Colonnes: {list(df_existant.columns)}")

print("\n📝 Aperçu:")
display(df_existant.head())

📂 DONNÉES EXISTANTES
Fichier: /home/henintsoa/CFIM/csv/marine_cotiere_2019_2020.csv
Nombre d'enregistrements: 1256
Colonnes: ['date', 'zone', 'vent', 'etat_mer', 'temps']

📝 Aperçu:


,date,zone,vent,etat_mer,temps
0,06/09/2019,CAP D'AMBRE A MAHANORO,10/15 kt atteignant 20/25 kt au nord d'Antalaha,agitée à forte,Pluies faible a modérée
1,06/09/2019,MAHANORO AU CAP SAINTE MARIE,05/10 kt devenant progressivement secteur sud ...,agitée à forte,pluies
2,06/09/2019,CAP D'AMBRE A BESALAMPY,"15/20 kt localement 20 kt au sud de Majunga, v...",Non spécifié,Temps sec
3,06/09/2019,BESALAMPY A MOROMBE,15/20 kt,"agitée a forte, très forte près Morombe dans l...",Temps sec
4,06/09/2019,MOROMBE A CAP SAINTE MARIE,20/25 kt atteignant 30/35 kt entre Morombe et,Non spécifié,Temps partiellement nuageux


In [8]:
# ============================================================
# FUSIONNER LES DONNÉES
# ============================================================

if len(df_meteo_extrait) > 0:
    # Harmoniser les colonnes
    df_nouveau = df_meteo_extrait[['date', 'zone', 'vent', 'etat_mer', 'temps']].copy()
    
    # Concaténer
    df_complet = pd.concat([df_existant, df_nouveau], ignore_index=True)
    
    # Supprimer les doublons (même date et zone)
    df_complet = df_complet.drop_duplicates(subset=['date', 'zone'], keep='first')
    
    # Trier par date
    df_complet['date_parsed'] = pd.to_datetime(df_complet['date'], format='%d/%m/%Y', errors='coerce')
    df_complet = df_complet.sort_values('date_parsed').reset_index(drop=True)
    
    print("📊 FUSION DES DONNÉES")
    print("=" * 70)
    print(f"Données existantes: {len(df_existant)} enregistrements")
    print(f"Nouvelles données: {len(df_nouveau)} enregistrements")
    print(f"Après fusion (sans doublons): {len(df_complet)} enregistrements")
    
    # Statistiques temporelles
    df_complet['annee'] = df_complet['date_parsed'].dt.year
    print("\n📅 RÉPARTITION PAR ANNÉE (après fusion):")
    print(df_complet['annee'].value_counts().sort_index())
else:
    df_complet = df_existant.copy()
    print("⚠️ Utilisation des données existantes uniquement")

📊 FUSION DES DONNÉES
Données existantes: 1256 enregistrements
Nouvelles données: 886 enregistrements
Après fusion (sans doublons): 1346 enregistrements

📅 RÉPARTITION PAR ANNÉE (après fusion):
annee
2019    498
2020    535
2021    142
2022    171
Name: count, dtype: int64


## 💾 7. Sauvegarde des Données Consolidées

In [9]:
# ============================================================
# SAUVEGARDE DU FICHIER CONSOLIDÉ
# ============================================================

# Colonnes finales à sauvegarder
colonnes_finales = ['date', 'zone', 'vent', 'etat_mer', 'temps']

# Créer une version propre pour la sauvegarde
df_final = df_complet[colonnes_finales].copy()

# Chemin de sortie
output_file = os.path.join(OUTPUT_PATH, 'meteo_marine_cotiere_complet.csv')

# Sauvegarder
df_final.to_csv(output_file, index=False, encoding='utf-8')

print("💾 SAUVEGARDE EFFECTUÉE")
print("=" * 70)
print(f"Fichier: {output_file}")
print(f"Nombre d'enregistrements: {len(df_final)}")
print(f"Taille: {os.path.getsize(output_file) / 1024:.1f} KB")

print("\n✅ Phase 1.1 terminée avec succès!")

💾 SAUVEGARDE EFFECTUÉE
Fichier: /home/henintsoa/CFIM/final/data/meteo_marine_cotiere_complet.csv
Nombre d'enregistrements: 1346
Taille: 182.3 KB

✅ Phase 1.1 terminée avec succès!


---

## 📋 8. Résumé et Prochaines Étapes

### Ce qui a été accompli:

1. ✅ **Inventaire complet** des bulletins météo disponibles (2019-2025)
2. ✅ **Extraction automatique** des données depuis les PDF/textes
3. ✅ **Parsing structuré** des zones côtières et conditions météo
4. ✅ **Consolidation** avec les données existantes
5. ✅ **Sauvegarde** du fichier consolidé

### Données produites:

| Fichier | Description |
|---------|-------------|
| `meteo_marine_cotiere_complet.csv` | Données météo consolidées 2019-2022 |

### Prochaines étapes:

➡️ **Notebook 2**: Création de la table de correspondance spatiale (zones ↔ régions)

➡️ **Notebook 3**: Standardisation des variables météo (conversion en valeurs numériques)

In [10]:
# ============================================================
# RÉSUMÉ FINAL
# ============================================================

print("\n" + "="*70)
print("                    RÉSUMÉ - PHASE 1.1")
print("="*70)

print("""
╔══════════════════════════════════════════════════════════════════════╗
║              EXTRACTION DES BULLETINS MÉTÉO - TERMINÉE              ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  📊 Données produites:                                               ║
║     • Fichier: meteo_marine_cotiere_complet.csv                     ║
║     • Enregistrements: voir ci-dessus                               ║
║     • Période: 2019-2022                                            ║
║                                                                      ║
║  📋 Variables extraites:                                             ║
║     • date: Date du bulletin                                        ║
║     • zone: Zone côtière                                            ║
║     • vent: Conditions de vent (texte)                              ║
║     • etat_mer: État de la mer (texte)                              ║
║     • temps: Conditions météo (texte)                               ║
║                                                                      ║
║  ➡️ PROCHAINE ÉTAPE:                                                 ║
║     Exécuter le notebook 2_correspondance_spatiale.ipynb            ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
""")


                    RÉSUMÉ - PHASE 1.1

╔══════════════════════════════════════════════════════════════════════╗
║              EXTRACTION DES BULLETINS MÉTÉO - TERMINÉE              ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  📊 Données produites:                                               ║
║     • Fichier: meteo_marine_cotiere_complet.csv                     ║
║     • Enregistrements: voir ci-dessus                               ║
║     • Période: 2019-2022                                            ║
║                                                                      ║
║  📋 Variables extraites:                                             ║
║     • date: Date du bulletin                                        ║
║     • zone: Zone côtière                                            ║
║     • vent: Conditions de vent (texte)                              ║
║     • etat_mer: É